In [17]:
from functions import *

In [18]:
a_vec = [763, 679, 397, 61, 697, 373, 
         289, 257, 625, 41, 193, 449]
b_vec = [435, 69, 330, 18, 612, 246, 
         496, 640, 200, 524, 672, 672] 

In [19]:
G = generate_g(a_vec)
all_indices = [i for i in range(l_h**2)]
gb_indices = [3, 8]
ga_indices = [i for i in all_indices if i not in gb_indices]
Ga = G.extract(ga_indices, list(range(G.cols)))
Gb = G.extract(gb_indices, list(range(G.cols)))
basis_a = solve_modular_kernel(Ga, P)
V = Matrix.hstack(*basis_a)

In [20]:
# 1. b_vec を行列形式に変換
b_mat = Matrix(b_vec)

# 2. V * x = b を有理数体（Q）上で解く
# V は main.ipynb で定義された Matrix.hstack(*basis_a)
try:
    # V が正則であれば x = V^-1 * b が求まる
    x_rational = V.solve(b_mat)

    # 3. 有理数の解 a/b を整数 a * inv(b, P) (mod P) に変換する関数
    def to_mod_p(val, p):
        num, den = val.as_numer_denom()
        # Python 3.8+ の pow(den, -1, p) はモジュラ逆数を計算する
        return (int(num) * pow(int(den), -1, p)) % p

    # 各要素に適用
    coefficients = x_rational.applyfunc(lambda v: to_mod_p(v, P))

    print("線形結合の係数ベクトル x:")
    display(coefficients)
    
    # 検算: V * x % P が b_vec と一致するか確認
    check = (V * coefficients).applyfunc(lambda x: x % P)
    if check == b_mat.applyfunc(lambda x: x % P):
        print("検算成功: 一致しました。")
    else:
        print("警告: 検算に失敗しました。")

except Exception as e:
    print(f"解を求めることができませんでした: {e}")

線形結合の係数ベクトル x:


Matrix([
[  3],
[709],
[689],
[746],
[762],
[732],
[ 68],
[ 84],
[565],
[557],
[744],
[346]])

検算成功: 一致しました。


In [21]:
cycles = generate_cycles(6)
h_x, h_z = generate_h_xz()
constraints = generate_constraints(cycles, a_vec, h_x, h_z)

In [22]:
# 全ての禁止ベクトル（法ベクトル）を個別にリスト化する
unique_forbidden_vectors = []
seen_vectors = set()

# 1. 条件B (潜在部の非可換性) からの制約 r_i
for i in range(Gb.rows):
    c_prime = (Gb.row(i) * V).applyfunc(lambda x: x % P)
    c_tuple = tuple(c_prime)
    if c_tuple not in seen_vectors:
        unique_forbidden_vectors.append(c_prime.T) # 列ベクトルとして保存
        seen_vectors.add(c_tuple)

# 2. 条件C (短いサイクルの回避) からの制約 c_prime
for c in constraints:
    c_prime = (Matrix([c]) * V).applyfunc(lambda x: x % P)
    c_tuple = tuple(c_prime)
    if c_tuple not in seen_vectors:
        unique_forbidden_vectors.append(c_prime.T)
        seen_vectors.add(c_tuple)

print(f"個別に回避すべき禁止制約（超平面）の数: {len(unique_forbidden_vectors)}")

個別に回避すべき禁止制約（超平面）の数: 312


In [23]:
def is_in_general_solution(x_vec, forbidden_vectors, p):
    """
    x_vec がすべての禁止超平面 r_i^T * x = 0 (mod p) を避けているか判定する。
    """
    # ベクトル形式を整える
    x_mat = Matrix(x_vec)
    
    for r in forbidden_vectors:
        # 内積が 0 (mod P) になったらその禁止領域に含まれている
        if (r.T * x_mat)[0] % p == 0:
            return False
    return True

# すでに見つけている特殊解 coefficients (x0) の妥当性を再確認
if is_in_general_solution(coefficients, unique_forbidden_vectors, P):
    print("特殊解 x0 は一般解の条件をすべて満たしています。")

特殊解 x0 は一般解の条件をすべて満たしています。


In [24]:
def find_another_solutions(x0, forbidden_vectors, p, count=5):
    """
    特殊解 x0 を起点に、条件を壊さない新しい解を生成する。
    """
    new_solutions = []
    da = x0.rows
    
    attempts = 0
    while len(new_solutions) < count and attempts < 1000000:
        # ランダムな摂動 delta_x を生成
        delta_x = Matrix([random.randint(0, p-1) for _ in range(da)])
        candidate = (x0 + delta_x).applyfunc(lambda x: x % p)
        
        if is_in_general_solution(candidate, forbidden_vectors, p):
            new_solutions.append(candidate)
        attempts += 1
        
    return new_solutions

# 一般解の集合から新しい 5 つの解を取得
general_solutions = find_another_solutions(coefficients, unique_forbidden_vectors, P, count=5)
print(f"{len(general_solutions)} 個の新しい一般解が見つかりました。")

0 個の新しい一般解が見つかりました。


In [25]:
# 生成された一般解の一つを b_vec に戻して確認
if general_solutions:
    sample_x = general_solutions[0]
    # b = V * x (mod P)
    sample_b = (V * sample_x).applyfunc(lambda val: val % P)
    
    print("一般解から得られた係数ベクトル b_vec:")
    print(list(sample_b))